# 06 | Player dependence and public content signals

**Author: Chanakya**

Test the risk in selling access around a popular player, using a clean historical results archive and bounded official-channel video metadata. Completed participation cannot guarantee future appearances.

In [ ]:
from pathlib import Path
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'data/manifests/release.json').exists())
sys.path.insert(0, str(ROOT))
from src.analysis_common import *
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo
rng = np.random.default_rng(CFG['seed'])
print('Offline inputs:', CFG['raw_release'], '| Author: Chanakya')
import src.analysis_common as shared
shared.ACTIVE_NOTEBOOK='06_player_and_video_risk'
shared.ACTIVE_SOURCES=['fancode_youtube_atp_search', 'fancode_youtube_tennis_search', 'sackmann_archive_2025']

## 1. Results audit and a conservative historical baseline
The 2026 alternative files have date/key inconsistencies. Use the documented 2025 archive for the quantitative player-risk baseline. Grand Slams and team events are excluded from the regular-tour comparison. The season file includes Brisbane and Hong Kong starting in December 2024. These are retained as 2025-season events, with original dates. United Cup is explicitly excluded even though its level code resembles a tour event. A player-event entry means at least one recorded match, not a complete entrants list including byes or withdrawn-before-play players.

In [ ]:
audit=pd.DataFrame(json.loads((ROOT/'data/manifests/results_source_qa.json').read_text()));display(table(audit,'06_results_source_audit'))
d=pd.read_csv(path('sackmann_archive_2025'));d['source_id']='sackmann_archive_2025'
# Remove byte-equivalent duplicate rows only, never a reused incomplete match key.
d=d.drop_duplicates();regular=d[d.tourney_level.isin(['A','M','F']) & ~d.tourney_name.str.contains('United Cup|Laver Cup',case=False,na=False)].copy();regular['tourney_date']=pd.to_datetime(regular.tourney_date.astype(str),format='%Y%m%d')
table(regular,'results_2025_regular',True)
appear=[]
for side in ['winner','loser']:
 z=regular[['tourney_id','tourney_name','tourney_date','round',side+'_name']].rename(columns={side+'_name':'player'});appear.append(z)
appear=pd.concat(appear,ignore_index=True)
rounds={'R128':1,'R64':2,'R32':3,'R16':4,'QF':5,'SF':6,'F':7,'RR':0}
appear['round_order']=appear['round'].map(rounds)
pe=appear.groupby(['player','tourney_id','tourney_name']).agg(matches=('round','size'),max_round=('round_order','max')).reset_index();pe['reached_SF']=pe.max_round>=6
players=pe.groupby('player').agg(events_played=('tourney_id','nunique'),matches=('matches','sum'),SF_events=('reached_SF','sum')).reset_index();players['SF_share_of_played_events']=players.SF_events/players.events_played
# Top ten by recorded match count is an explicit, reproducible retrospective selection rule.
top=players.nlargest(10,'matches');display(table(top,'06_player_progression'));table(pe,'player_event_2025',True)
plt.figure(figsize=(10,4));plt.barh(top.sort_values('SF_share_of_played_events').player,top.sort_values('SF_share_of_played_events').SF_share_of_played_events*100);plt.xlabel('Percent of recorded played events reaching semifinal');plt.title('Even busy players do not guarantee a weekend appearance');fig('06_player_appearance_risk','2025 regular-tour archive. Retrospective top ten by matches, not popular-player WTP or 2026 prediction.')
# Compare single-player and two-player coverage within observed event universe.
universe=set(regular.tourney_id);names=top.player.iloc[:3].tolist();bundle=[]
from itertools import combinations
for k in [1,2,3]:
 for chosen in combinations(names,k):
  x=pe[pe.player.isin(chosen)];bundle.append(dict(players=' + '.join(chosen),events_with_any_recorded_appearance=x.tourney_id.nunique(),events_with_any_SF=x[x.reached_SF].tourney_id.nunique(),archive_event_universe=len(universe)))
display(table(pd.DataFrame(bundle),'06_player_bundle_coverage'))

## 2. Parse bounded FanCode video-search metadata
Deduplicate video IDs across two ranked keyword searches. Views are displayed cumulative counts, not India-specific viewers. Relative publication labels do not support exact views-per-day normalization. Do not correlate publication hour with match start.

In [ ]:
videos={}
def walk(obj):
 if isinstance(obj,dict):
  for key,value in obj.items():
   if key in ['videoRenderer','channelVideoPlayerRenderer']:yield value
   yield from walk(value)
 elif isinstance(obj,list):
  for value in obj:yield from walk(value)
def label(x):return x.get('simpleText',''.join(r.get('text','') for r in x.get('runs',[])))
for sid in ['fancode_youtube_tennis_search','fancode_youtube_atp_search']:
 raw=path(sid).read_text();m=re.search(r'(?:var )?ytInitialData\s*=\s*',raw);obj,_=json.JSONDecoder().raw_decode(raw[m.end():])
 assert obj['metadata']['channelMetadataRenderer']['externalId']=='UCF10AG_t1AYW3mlmX7g1VJA'
 for v in walk(obj.get('contents',{})):
  vid=v['videoId'];title=label(v.get('title',{}));views=label(v.get('viewCountText',{}));age=label(v.get('publishedTimeText',{}))
  count=re.search(r'([\d,]+) views?',views);videos[vid]=dict(video_id=vid,title=title,displayed_views=views,views=int(count.group(1).replace(',','')) if count else np.nan,publication_label=age,source_id=sid)
v=pd.DataFrame(videos.values());v['star_mention']=v.title.str.contains('Alcaraz|Sinner|Djokovic|Nadal',case=False,regex=True);v['highlight_mention']=v.title.str.contains('highlight',case=False)
table(v,'video_metadata',True);display(table(v.nlargest(15,'views'),'06_top_displayed_videos'))
summary=v.groupby('star_mention').agg(videos=('video_id','size'),median_views=('views','median'),total_views=('views','sum')).reset_index();display(table(summary,'06_video_star_diagnostic'))
plt.figure(figsize=(8,4))
for k,g in v.groupby('star_mention'):plt.scatter(np.repeat(int(k),len(g))+rng.normal(0,.025,len(g)),np.log10(g.views+1),alpha=.55,label='Star-name title' if k else 'Other title')
plt.xticks([0,1],['Other title','Selected star mentioned']);plt.ylabel('log10(displayed views + 1)');plt.title('Content popularity is highly dispersed');fig('06_video_views','Keyword-selected metadata, cumulative views, mixed ages and competitions. Title association is not player causal effect.')
check('06_player_video',{'video_ids_unique':v.video_id.is_unique,'video_count_53':len(v)==53,'only_2025_season_ids':bool(regular.tourney_id.str.startswith('2025-').all()),'player_event_unique':not pe.duplicated(['player','tourney_id']).any(),'progression_bounded':bool(players.SF_share_of_played_events.between(0,1).all())})
report('06_player_video_findings','Prefer event or flexible access over an unconditional player-final promise. Historical played-event progression shows appearance risk and does not count pre-event withdrawals. Videos provide a creative prioritization proxy, not willingness to pay or a price premium. Replays, alternative-player options and clear refunds should be evaluated before testing a player-specific SKU.')

## Source references
These IDs resolve to the preserved bodies, URLs and capture timestamps. Derived tables also retain row-level source IDs where applicable. Case inputs refer to the supplied brief, physical PDF pages 9–14. Review source files resolve through the review collection log. Scenario parameters are in analysis_config.

In [ ]:
references=source_table(['fancode_youtube_atp_search', 'fancode_youtube_tennis_search', 'sackmann_archive_2025'])
display(table(references,'06_source_references'))